# Early Prediction of Knee Osteoarthritis Progression Using Machine Learning
## Notebook 01: Exploratory Data Analysis & Baseline Modeling

**Project Goal**: Build a research prototype to predict the risk of knee osteoarthritis (OA) progression using baseline clinical/tabular features from the Osteoarthritis Initiative (OAI) dataset.

---

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Ensure project root is in sys.path
sys.path.append('..')

from src.config import config
from src.inspect_dataset import run_stage_1_inspection
from src.train import create_demo_synthetic_dataset, train_models
from src.feature_engineering import create_progression_target, enforce_baseline_features_only
from src.evaluate import compute_medical_metrics

## 1. Stage 1: Dataset Inspection & Schema Profile
Scanning `data/raw/` to profile raw OAI files.

In [ ]:
profiles = run_stage_1_inspection(Path('../data/raw'), Path('../reports/results/oai_schema_report.md'))

## 2. Load Dataset & Target Construction
If no raw OAI file exists in `data/raw/`, we utilize the schema-compliant synthetic cohort for demonstration.

In [ ]:
raw_csv = Path('../data/processed/oai_processed.csv')
if raw_csv.exists():
    df = pd.read_csv(raw_csv)
else:
    print('Generating synthetic demo dataset matching OAI baseline schema...')
    df = create_demo_synthetic_dataset(num_samples=1000)

if config.TARGET_COL not in df.columns:
    df = create_progression_target(df)

df.head()

## 3. Exploratory Data Analysis (EDA)

### 3.1 Class Balance

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=config.TARGET_COL, palette='Set2')
plt.title('Target Class Distribution (0: No Progression, 1: Progression)')
plt.xlabel('Progression Status')
plt.ylabel('Patient Count')
plt.show()

### 3.2 Age & BMI Distributions by Progression Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df, x='V00AGE', hue=config.TARGET_COL, common_norm=False, ax=axes[0], fill=True)
axes[0].set_title('Age Distribution by OA Progression Status')

sns.kdeplot(data=df, x='V00BMI', hue=config.TARGET_COL, common_norm=False, ax=axes[1], fill=True)
axes[1].set_title('BMI Distribution by OA Progression Status')

plt.tight_layout()
plt.show()

### 3.3 WOMAC Pain Score vs Baseline KL Grade

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x='V00KL', y='V00WOMKP', hue=config.TARGET_COL, palette='coolwarm')
plt.title('Baseline WOMAC Pain Score by Baseline KL Grade and Progression Outcome')
plt.xlabel('Baseline Kellgren-Lawrence (KL) Grade')
plt.ylabel('Baseline WOMAC Pain Score')
plt.show()

## 4. Execute Pipeline Training & Evaluation

In [ ]:
best_model, eval_results = train_models(df)